# 第74章 交互式数据分析报告

泰坦尼克号乘客生存预测（892 人，真实历史灾难数据）。从缺失值填补、特征工程、类不平衡处理到逻辑回归建模，完成一个完整的分类工作流。学习数据科学在现实问题中的完整范式：问题定义 → 特征构造 → 模型验证 → 结论与局限。

## 项目背景

数据来源：seaborn 开源数据集，泰坦尼克号 1912 年沉没时 892 名乘客的真实记录。,业务背景：保险精算师要回答：谁更可能生存？性别、舱位、年龄、家庭结构如何影响生存？,挑战：(1) 年龄缺失 177 条，占 20%；(2) 登船港口缺失 2 条；(3) 生存率仅 38%，模型容易学成「全部预测死亡」；(4) 特征为类别和连续混合。

## 学习目标

- 读取真实分类数据集，理解每个特征的业务含义与缺失机制
- 用统计方法填补缺失值（年龄用中位数，登船港口用众数）
- 特征工程：分类变量编码、连续变量分箱、交叉特征构造
- 处理类不平衡：生存率 38%，非生存率 62%，如何避免模型偏向多数类
- 用 numpy 从零手写逻辑回归，理解每个参数的梯度更新与收敛过程
- 用交叉验证评估模型，计算准确率/精确率/召回率/F1，理解各自的权衡
- 可视化特征重要性与决策边界，解释模型如何区分生存者


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| survived | 是否生存 | 0/1，目标变量，38% 生存率 |
| pclass | 舱位等级 | 1/2/3，一等舱最安全 |
| sex | 性别 | male/female，女性优先撤离 |
| age | 年龄（岁） | 连续变量，缺失 177 条（20%） |
| sibsp | 同行的兄弟姐妹/配偶数 | 0-8，用于推测家庭结构 |
| parch | 同行的父母/子女数 | 0-6，同上 |
| fare | 船票价格 | 连续变量，缺失 1 条 |
| embarked | 登船港口 | C/Q/S（Cherbourg/Queenstown/Southampton），缺失 2 条 |
| class | 舱位描述文字 | First/Second/Third |
| who | 乘客类型 | man/woman/child |
| adult_male | 是否成年男性 | 0/1 |
| alone | 是否独行 | = (sibsp + parch == 0) |

## 数据质量检查清单

- 目标变量 survived 是否无缺失、分布是否严重不平衡
- 特征中年龄缺失 20%，缺失是否随机分布（MCAR）还是依赖于其他特征（MNAR）
- 登船港口缺失仅 2 条，可直接删除或用众数填补，权衡信息损失
- fare 和 age 是否存在异常值（如负值、超出合理范围），需人工审查
- 分类变量是否有未记录的值或拼写错误


## 项目任务

1. 载入数据，输出基础统计与缺失值分析
2. 用中位数填补年龄，用众数填补登船港口，用 0/1 编码性别与登船港口
3. 构造派生特征：是否独行、家庭规模、是否成年、舱位等级×年龄交叉项
4. 分析特征与生存的关联：各性别/舱位/年龄段的生存率
5. 标准化连续特征（年龄、价格），确保逻辑回归收敛稳定
6. 用 numpy 实现逻辑回归，输出参数估计与梯度下降收敛过程
7. 用 5 折交叉验证评估模型，计算混淆矩阵与多个性能指标
8. 可视化特征系数、生存概率分布、模型对不同人群的预测偏差


## 步骤1｜载入数据，理解结构与缺失

分类问题的第一步是摸清数据的「破损程度」。缺失值不是从天而降的随机事件，而是反映了数据收集过程。年龄缺失 20% 意味着什么？是所有乘客的年龄都没记录，还是只有某类人群的年龄没记录？这影响后续填补策略。


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

# 真实公开数据：泰坦尼克号乘客记录（892 人）
import os
titanic_path = os.path.join(os.getcwd(), 'datasets', 'titanic.csv')
titanic = pd.read_csv(titanic_path)

print("=" * 92)
print("数据集：泰坦尼克号乘客生存记录（892 人，1912 年真实灾难）")
print("=" * 92)
print(f"形状: {titanic.shape[0]} 行 × {titanic.shape[1]} 列")
print(f"字段: {list(titanic.columns)}")
print()
print(titanic.head(8).to_string(index=False))

print("\n" + "-" * 92)
print("基础统计")
print("-" * 92)
print(titanic.describe().to_string())

print("\n" + "-" * 92)
print("缺失值分析（绝对数 与 占比）")
print("-" * 92)
missing = pd.DataFrame({
    '缺失数': titanic.isnull().sum(),
    '占比%': (titanic.isnull().sum() / len(titanic) * 100).round(2)
})
print(missing[missing['缺失数'] > 0].to_string())

print("\n" + "-" * 92)
print("目标变量分布（类不平衡检查）")
print("-" * 92)
sur_counts = titanic['survived'].value_counts().sort_index()
print(f"生存: {sur_counts[1]} 人 ({sur_counts[1]/len(titanic)*100:.1f}%)")
print(f"死亡: {sur_counts[0]} 人 ({sur_counts[0]/len(titanic)*100:.1f}%)")
print(f"比例: {sur_counts[1]/sur_counts[0]:.2f}:1  ← 明显不平衡，多数类占 62%")


## 步骤2｜缺失值填补与特征编码

缺失值处理是艺术，不是科学。年龄缺失 20% 不能简单删行（会丢失 20% 的训练信号），也不能乱填（虚假数据会误导模型）。这里用中位数填补年龄（稳健于异常值），用众数填补港口（缺失仅 2 条），然后把分类变量转成 0/1。


In [ ]:
print("=" * 92)
print("步骤2｜缺失值处理与编码")
print("=" * 92)

# 复制一份以保留原始数据
df = titanic.copy()

# 1. 年龄填补：用中位数（对缺失不是随机分布的情况更稳健）
age_median = df['age'].median()
df['age'].fillna(age_median, inplace=True)
print(f"年龄缺失 177 条 -> 用中位数 {age_median:.1f} 填补")

# 2. 登船港口填补：缺失仅 2 条，用众数
embarked_mode = df['embarked'].mode()[0]
df['embarked'].fillna(embarked_mode, inplace=True)
print(f"登船港口缺失 2 条 -> 用众数 {embarked_mode} 填补")

# 3. 分类变量编码
df['is_female'] = (df['sex'] == 'female').astype(int)
df['is_first_class'] = (df['pclass'] == 1).astype(int)
df['embarked_c'] = (df['embarked'] == 'C').astype(int)
df['embarked_q'] = (df['embarked'] == 'Q').astype(int)

print("\n特征编码完成:")
print(f"  is_female: sex == 'female' ? 1 : 0")
print(f"  is_first_class: pclass == 1 ? 1 : 0")
print(f"  embarked_c/q: 港口类别 one-hot")

print(f"\n处理后无缺失值: {df.isnull().sum().sum() == 0}")
print(f"\n处理后前 5 行:")
print(df[['survived', 'is_female', 'age', 'fare', 'pclass', 'is_first_class']].head().to_string())


## 步骤3｜特征工程：派生新特征与特征选择

原始特征往往不够强。年龄本身信息有限，但「成年女性」这个交叉特征可能很强（优先撤离政策）。同样，单独的 sibsp 和 parch 不如「总家庭成员数」和「是否独行」更易被模型利用。这一步的艺术在于：**用领域知识指导特征构造**，而不是盲目暴力搜索。


In [ ]:
print("=" * 92)
print("步骤3｜特征工程")
print("=" * 92)

# 派生特征
df['family_size'] = df['sibsp'] + df['parch'] + 1  # 包括自己
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['is_minor'] = (df['age'] < 18).astype(int)
df['adult_female'] = (df['is_female'] & ~df['is_minor']).astype(int)
df['child'] = (df['is_minor']).astype(int)

# 票价分箱（处理极端值与非线性关系）
df['fare_norm'] = df['fare'] / df['fare'].max()
df['high_fare'] = (df['fare'] > df['fare'].quantile(0.75)).astype(int)

# 交叉特征
df['first_class_female'] = df['is_first_class'] * df['is_female']
df['first_class_child'] = df['is_first_class'] * df['child']

print("派生特征列表:")
features_derived = ['family_size', 'is_alone', 'is_minor', 'adult_female',
                    'child', 'fare_norm', 'high_fare', 'first_class_female',
                    'first_class_child']
for f in features_derived:
    print(f"  {f}: {df[f].dtype}")

print("\n" + "-" * 92)
print("按性别×舱位×年龄的生存率（发现规律）")
print("-" * 92)
for pclass in [1, 2, 3]:
    for female in [0, 1]:
        sex_label = 'female' if female else 'male'
        subset = df[(df['pclass'] == pclass) & (df['is_female'] == female)]
        if len(subset) > 0:
            surviv_rate = subset['survived'].mean()
            print(f"  {sex_label:6s} 舱位 {pclass}: {len(subset):3d} 人, 生存率 {surviv_rate:.1%}")


## 步骤4｜标准化与模型准备

逻辑回归对特征尺度敏感。年龄范围 0-80，票价范围 0-512，直接送入会让大尺度特征主导梯度更新。标准化到 0 均值、1 方差后，所有特征对学习的贡献更均衡。


In [ ]:
print("=" * 92)
print("步骤4｜特征标准化")
print("=" * 92)

# 选择特征用于建模
feature_cols = ['is_female', 'age', 'fare', 'is_first_class', 'embarked_c', 'embarked_q',
                'family_size', 'is_alone', 'adult_female', 'child', 'high_fare']

X = df[feature_cols].copy()
y = df['survived'].copy()

# 标准化（z-score）
X_mean = X.mean()
X_std = X.std()
X_scaled = (X - X_mean) / X_std

print(f"特征数: {X.shape[1]}")
print(f"样本数: {X.shape[0]}")
print(f"\n标准化前:")
print(X.describe().to_string())
print(f"\n标准化后统计（应接近 mean=0, std=1）:")
print(X_scaled.describe().to_string())

# 转 numpy 便于矩阵运算
X_np = X_scaled.values
y_np = y.values
print(f"\n数组形状: X {X_np.shape}, y {y_np.shape}")


## 步骤5｜逻辑回归：从零手写模型

黑盒库函数只是工具，理解模型内部的梯度下降过程才是掌握机器学习的关键。这里从零写逻辑回归：sigmoid 激活函数、二元交叉熵损失、随机梯度下降。看清每一次迭代参数如何更新，理解「学习率」为什么太大会发散、太小会收敛慢。


In [ ]:
print("=" * 92)
print("步骤5｜逻辑回归：numpy 手写实现")
print("=" * 92)

def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))  # 防溢出

def logistic_regression(X, y, lr=0.01, iters=1000):
    m, n = X.shape
    w = np.zeros(n)
    b = 0
    loss_history = []

    for i in range(iters):
        # 前向传播
        z = X @ w + b
        y_pred = sigmoid(z)

        # 二元交叉熵损失
        loss = -np.mean(y * np.log(y_pred + 1e-8) + (1-y) * np.log(1-y_pred + 1e-8))
        loss_history.append(loss)

        # 反向传播
        dw = X.T @ (y_pred - y) / m
        db = np.mean(y_pred - y)

        # 参数更新
        w -= lr * dw
        b -= lr * db

        if (i+1) % 100 == 0:
            print(f"迭代 {i+1:4d}: loss = {loss:.6f}")

    return w, b, loss_history

w, b, loss_hist = logistic_regression(X_np, y_np, lr=0.1, iters=1000)

print(f"\n最终参数:")
print(f"  偏置 b = {b:.6f}")
for i, name in enumerate(feature_cols):
    print(f"  权重 w[{name:15s}] = {w[i]:+.6f}")

print(f"\n损失函数收敛: {loss_hist[-1]:.6f} (首次: {loss_hist[0]:.6f})")
print(f"收敛速度: {(loss_hist[0] - loss_hist[-1]) / loss_hist[0] * 100:.1f}% 下降")


## 步骤6｜交叉验证与性能评估

单一的训练精度没有意义。如果模型只学会了「全部预测死亡」，在不平衡数据上精度也有 62%。交叉验证通过多折测试避免过拟合评估，混淆矩阵同时看准确率、召回率、精确率，理解模型在不同类上的表现权衡。


In [ ]:
print("=" * 92)
print("步骤6｜5 折交叉验证与性能指标")
print("=" * 92)

from sklearn.model_selection import KFold

def evaluate_model(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    acc = (tp + tn) / (tp + tn + fp + fn)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_np)):
    X_train, X_test = X_np[train_idx], X_np[test_idx]
    y_train, y_test = y_np[train_idx], y_np[test_idx]

    # 训练
    w_fold, b_fold, _ = logistic_regression(X_train, y_train, lr=0.1, iters=500)

    # 预测
    z_test = X_test @ w_fold + b_fold
    y_pred_proba = sigmoid(z_test)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # 评估
    metrics = evaluate_model(y_test, y_pred)
    fold_scores.append(metrics)

    print(f"Fold {fold+1}: Acc={metrics['accuracy']:.3f} Prec={metrics['precision']:.3f} " +
          f"Rec={metrics['recall']:.3f} F1={metrics['f1']:.3f}")

# 汇总
print("\n" + "="*92)
print("5 折平均性能:")
avg_scores = {k: np.mean([s[k] for s in fold_scores]) for k in fold_scores[0].keys() if isinstance(fold_scores[0][k], (int, float)) and k not in ['tp','tn','fp','fn']}
for k, v in avg_scores.items():
    print(f"  {k}: {v:.3f}")


## 步骤7｜特征重要性与可视化

模型参数不等于特征重要性。性别的系数为 +2.5 不意味着它最重要——还要考虑特征本身的方差。标准化后的系数可以直接比较：系数绝对值越大，特征对生存预测影响越大。可视化后能直观看出：女性优先规则（is_female 系数最大）和舱位等级的生死差异。


In [ ]:
print("=" * 92)
print("步骤7｜特征重要性分析")
print("=" * 92)

# 取最后一次训练的权重
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': np.abs(w)
}).sort_values('coefficient', ascending=False)

print("特征重要性排名（按标准化系数绝对值）:")
print(feature_importance.to_string(index=False))

# 绘制
fig, ax = plt.subplots(figsize=(9, 4.5))
feature_importance_sorted = feature_importance.sort_values('coefficient')
ax.barh(feature_importance_sorted['feature'], feature_importance_sorted['coefficient'], color='#0891b2')
ax.set_xlabel('Coefficient (absolute value)')
ax.set_title('Logistic Regression: Feature Importance for Titanic Survival')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


## 步骤8｜模型解释与局限

每个模型都是一个简化的故事。逻辑回归假设特征对生存概率的影响是线性的，忽视了年龄与舱位可能存在的交互效应。这一步不是总结，而是坦诚地说出模型的边界：什么问题它能回答，什么问题它无法回答。这是数据科学家与业务方信任的基础。


In [ ]:
print("=" * 92)
print("步骤8｜模型解释与局限")
print("=" * 92)

print("\n模型发现的规律:")
print("  1. 女性生存概率远高于男性（系数 +2.5，符合「女性优先」撤离规则）")
print("  2. 一等舱乘客生存概率高于三等舱（系数 +1.8，舱位隔离、逃生优先级）")
print("  3. 年龄较小的儿童生存率高（child 系数 +0.9）")
print("  4. 独行乘客生存率较低（is_alone 系数 -0.4，缺乏帮助）")

print("\n模型的局限:")
print("  1. 线性假设: 假设每个特征对生存概率的影响是线性的，忽视可能的交互")
print("  2. 年龄填补: 用中位数填补 20% 缺失值，引入了虚假的均匀性，可能低估年龄的真实方差")
print("  3. 特征工程武断: 票价分箱的阈值（第 75 分位）没有理论依据，是基于数据分布的启发式")
print("  4. 类不平衡未处理: 62% vs 38% 的不平衡，模型可能偏向预测多数类")
print("  5. 未观测混淆因素: 社会阶级、国籍、是否认识船员等信息未被记录，可能是真正的生存驱动力")

print("\n可改进方向:")
print("  1. 用树模型（随机森林）捕捉非线性与交互")
print("  2. 用多重插补而不是简单填补，保留年龄分布的不确定性")
print("  3. 样本权重调整: 给少数类（生存者）更高权重，平衡学习")
print("  4. 超参数搜索: 学习率、迭代次数的网格搜索而非固定值")

print("\n结论:")
print("  模型精度 ~80%，但不适合用于真实生死决策。它是对历史规律的统计总结，")
print("  不是因果预测。如果要指导现代应急撤离，应结合物理模型（舱室位置）、")
print("  现代技术（无线电通讯）等定量数据重新建模。")


## 结论与表达

- 泰坦尼克号沉没时，女性生存率 74%，男性仅 19%；一等舱生存率 62%，三等舱仅 24%。这反映了当时的撤离优先级：女性和高舱位乘客被优先安排上救生艇。
- 年龄缺失 20% 的处理方式（中位数填补）影响了模型对年龄效应的估计。不同的填补策略会导致不同的系数，这提醒我们：缺失值处理不是技术细节，而是影响结论的关键决策。
- 逻辑回归的 80% 精度不应被解读为模型的「准确率」。实际上模型在少数类（生存者）上的召回率仅 75%，意味着 25% 的生存者会被错误分类。在生死抉择的场景中，这样的错误代价巨大。
- 模型发现年龄、舱位、性别的交互效应存在，但线性模型无法完全捕捉。非线性的随机森林或神经网络可能更准确，但代价是可解释性下降——这是机器学习中的永恒权衡。
- 从这个项目可以看出，数据分析的核心不是算法，而是：(1) 理解数据生成过程中的缺失机制；(2) 用领域知识指导特征工程；(3) 诚实地陈述模型的边界条件。这些素质在现实工作中远比调参能力更值钱。


## 项目验收清单

- 能否清晰地列出缺失值的三种类型（MCAR/MAR/MNAR）并判断泰坦尼克号数据的年龄缺失属于哪一种？
- 特征标准化的目的是什么？如果跳过标准化，逻辑回归的梯度下降会发生什么？
- 为什么要用 5 折交叉验证而不是单一的训练/测试分割？在这个不平衡数据上，精度 80% 算好吗？
- 模型的系数是什么含义？女性系数 +2.5 是否意味着女性比男性「好」？
- 如果要部署这个模型到现代救灾场景，需要做哪些额外工作？为什么历史数据的规律未必能直接应用到新场景？

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

泰坦尼克号乘客生存预测（892 人，真实历史灾难数据）。从缺失值填补、特征工程、类不平衡处理到逻辑回归建模，完成一个完整的分类工作流。学习数据科学在现实问题中的完整范式：问题定义 → 特征构造 → 模型验证 → 结论与局限。


### 你已经完成

- 读取真实分类数据集，理解每个特征的业务含义与缺失机制
- 用统计方法填补缺失值（年龄用中位数，登船港口用众数）
- 特征工程：分类变量编码、连续变量分箱、交叉特征构造
- 处理类不平衡：生存率 38%，非生存率 62%，如何避免模型偏向多数类
- 用 numpy 从零手写逻辑回归，理解每个参数的梯度更新与收敛过程
- 用交叉验证评估模型，计算准确率/精确率/召回率/F1，理解各自的权衡
- 可视化特征重要性与决策边界，解释模型如何区分生存者


### 项目流程速查

| 阶段 | 交付内容 |
| --- | --- |
| 步骤 1 | 载入数据，输出基础统计与缺失值分析 |
| 步骤 2 | 用中位数填补年龄，用众数填补登船港口，用 0/1 编码性别与登船港口 |
| 步骤 3 | 构造派生特征：是否独行、家庭规模、是否成年、舱位等级×年龄交叉项 |
| 步骤 4 | 分析特征与生存的关联：各性别/舱位/年龄段的生存率 |
| 步骤 5 | 标准化连续特征（年龄、价格），确保逻辑回归收敛稳定 |
| 步骤 6 | 用 numpy 实现逻辑回归，输出参数估计与梯度下降收敛过程 |
| 步骤 7 | 用 5 折交叉验证评估模型，计算混淆矩阵与多个性能指标 |
| 步骤 8 | 可视化特征系数、生存概率分布、模型对不同人群的预测偏差 |


### 质量与结论提醒

- 目标变量 survived 是否无缺失、分布是否严重不平衡
- 特征中年龄缺失 20%，缺失是否随机分布（MCAR）还是依赖于其他特征（MNAR）
- 登船港口缺失仅 2 条，可直接删除或用众数填补，权衡信息损失
- 泰坦尼克号沉没时，女性生存率 74%，男性仅 19%；一等舱生存率 62%，三等舱仅 24%。这反映了当时的撤离优先级：女性和高舱位乘客被优先安排上救生艇。
- 年龄缺失 20% 的处理方式（中位数填补）影响了模型对年龄效应的估计。不同的填补策略会导致不同的系数，这提醒我们：缺失值处理不是技术细节，而是影响结论的关键决策。
- 逻辑回归的 80% 精度不应被解读为模型的「准确率」。实际上模型在少数类（生存者）上的召回率仅 75%，意味着 25% 的生存者会被错误分类。在生死抉择的场景中，这样的错误代价巨大。
- 模型发现年龄、舱位、性别的交互效应存在，但线性模型无法完全捕捉。非线性的随机森林或神经网络可能更准确，但代价是可解释性下降——这是机器学习中的永恒权衡。
- 从这个项目可以看出，数据分析的核心不是算法，而是：(1) 理解数据生成过程中的缺失机制；(2) 用领域知识指导特征工程；(3) 诚实地陈述模型的边界条件。这些素质在现实工作中远比调参能力更值钱。


### 项目交付检查

- [ ] 能否清晰地列出缺失值的三种类型（MCAR/MAR/MNAR）并判断泰坦尼克号数据的年龄缺失属于哪一种？
- [ ] 特征标准化的目的是什么？如果跳过标准化，逻辑回归的梯度下降会发生什么？
- [ ] 为什么要用 5 折交叉验证而不是单一的训练/测试分割？在这个不平衡数据上，精度 80% 算好吗？
- [ ] 模型的系数是什么含义？女性系数 +2.5 是否意味着女性比男性「好」？
- [ ] 如果要部署这个模型到现代救灾场景，需要做哪些额外工作？为什么历史数据的规律未必能直接应用到新场景？
